In [ ]:
#%matplotlib inline
import argparse
import os
import random
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.utils.data
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

import dac
import torchaudio
from IPython.display import Audio

In [2]:
dataroot = "training_data"
target_dir = "beatbox"
corpus_dir = "drum_kit"


workers = 1
batch_size = 2
frame_length = 320
nz = 1024
ngf = 1024
ndf = 64
num_epochs = 5
lr = 0.0002
beta1 = 0.5
ngpu = 0

In [3]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

In [4]:
# Load the audio file

def load_mono(file_name, target_sr):
    audio, sr = torchaudio.load(file_name)
    if audio.shape[0] > 1:
        audio = audio.mean(0, keepdim=True)

    if sr != target_sr:
        audio = torchaudio.functional.resample(audio, sr, target_sr)
        sr = target_sr
    audio = audio.clamp(-1, 1)
    return audio

In [5]:
class OneStepGenerator(nn.Module):
    # generate 1024d embedding from 1024d embedding
    def __init__(self):
        super(OneStepGenerator, self).__init__()
        self.main = nn.Sequential(
            nn.Linear(1024, 256),
            nn.LeakyReLU(),
            nn.Linear(256, 64),
            nn.LeakyReLU(),
            nn.Linear(64, 256),
            nn.LeakyReLU(),
            nn.Linear(256, 1024)
        )

    def forward(self, input):
        return self.main(input)

class BlockGenerator(nn.Module):
    def __init__(self):
        super(BlockGenerator, self).__init__()
        self.main = nn.Sequential(
            nn.Conv1d(1024, 256, kernel_size=5, padding="same", padding_mode="reflect"),
            nn.LeakyReLU(),
            nn.Conv1d(256, 64, kernel_size=5, padding="same", padding_mode="reflect"),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(),
            nn.Conv1d(64, 256, kernel_size=5, padding="same", padding_mode="reflect"),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(),
            nn.Conv1d(256, 1024, kernel_size=5, padding="same", padding_mode="reflect")
        )

    def forward(self, input):
        return self.main(input)

class OneStepDiscriminator(nn.Module):
    # generate discriminator score between 0 and 1.
    def __init__(self):
        super(OneStepDiscriminator, self).__init__()
        self.embedding_path = nn.Sequential(
            nn.Linear(1024, 256),
            nn.LeakyReLU(),
            nn.Linear(256, 64),
            nn.LeakyReLU(),
            nn.Linear(64, 256),
            nn.LeakyReLU(),
            nn.Linear(256, 64)
        )

        self.waveform_path = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(),
            nn.Conv1d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Conv1d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(64),
            nn.LeakyReLU()
        )

        self.combined_layer = nn.Sequential(
            nn.Linear(128, 512),
            nn.LeakyReLU(),
            nn.Linear(512, 1),
            nn.Sigmoid()
        )
    
    def forward(self, embedding, waveform):
        embedding_features = self.embedding_path(embedding)
        waveform_features = self.waveform_path(waveform)

        combined_features = torch.cat([embedding_features, waveform_features])
        output_probability = self.combined_layer(combined_features)

        return output_probability 

class BlockDiscriminator(nn.Module):
    # generate discriminator score between 0 and 1.
    def __init__(self, block_length_in_samples, block_length_in_frames):
        super(BlockDiscriminator, self).__init__()

        self.block_length_in_samples = block_length_in_samples
        self.block_length_in_frames = block_length_in_frames

        # FIX THESE SO THAT IT CAN CONVERT A SEQUENCE OF EMBEDS + A WAVEFORM TO A SCALAR PROBABILITY
        self.embedding_path = nn.Sequential(
            nn.Conv1d(1024, 256, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(),
            nn.Conv1d(256, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(),
        )

        self.waveform_path = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(),
            nn.Conv1d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Conv1d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(64),
            nn.LeakyReLU()
        )

        self.combined_layer = nn.Sequential(
            nn.Linear((64*(block_length_in_frames//4) + 64*(block_length_in_samples//8)), 512),
            nn.LeakyReLU(),
            nn.Linear(512, 1),
            nn.Sigmoid()
        )
    
    def forward(self, embedding, waveform):
        embedding = embedding.to(torch.float32)
        waveform = waveform.to(torch.float32)

        embedding_features = self.embedding_path(embedding)
        waveform_features = self.waveform_path(waveform)

        combined_features = torch.cat([embedding_features.flatten(1, -1), waveform_features.flatten(1, -1)], dim=1)
        output_probability = self.combined_layer(combined_features)


        return output_probability 

In [6]:
def trim_silence(input, sample_rate):
    # Compute root mean square (RMS) energy per frame
    frame_size = int(sample_rate * 0.05)
    rms_energy = input.unfold(1, frame_size, frame_size).pow(2).mean(dim=2).sqrt()

    # Set silence threshold (e.g., 10% of max RMS energy)
    threshold = 0.1 * rms_energy.max()

    # Find frames where energy is above the threshold
    non_silent_indices = (rms_energy > threshold).nonzero(as_tuple=True)[1]

    trimmed_waveform = torch.zeros([1,1])
    for index in non_silent_indices:
        trimmed_waveform = torch.cat([trimmed_waveform, input[:,index*frame_size:(index+1)*frame_size]], dim=1)
    
    return trimmed_waveform

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"

dac_model = dac.DAC.load(dac.utils.download()).to(device)
print(dac_model)

/home/mille/miniconda3/envs/dac/lib/python3.13/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


DAC(
  (encoder): Encoder(
    (block): Sequential(
      (0): Conv1d(1, 64, kernel_size=(7,), stride=(1,), padding=(3,))
      (1): EncoderBlock(
        (block): Sequential(
          (0): ResidualUnit(
            (block): Sequential(
              (0): Snake1d()
              (1): Conv1d(64, 64, kernel_size=(7,), stride=(1,), padding=(3,))
              (2): Snake1d()
              (3): Conv1d(64, 64, kernel_size=(1,), stride=(1,))
            )
          )
          (1): ResidualUnit(
            (block): Sequential(
              (0): Snake1d()
              (1): Conv1d(64, 64, kernel_size=(7,), stride=(1,), padding=(9,), dilation=(3,))
              (2): Snake1d()
              (3): Conv1d(64, 64, kernel_size=(1,), stride=(1,))
            )
          )
          (2): ResidualUnit(
            (block): Sequential(
              (0): Snake1d()
              (1): Conv1d(64, 64, kernel_size=(7,), stride=(1,), padding=(27,), dilation=(9,))
              (2): Snake1d()
              

In [ ]:
target_sr = dac_model.sample_rate

tempo = 90
subdivision = 8 # subdivision = 8 ==> 1/8th note blocks

block_length_in_samples = int(target_sr*60/(tempo*subdivision/4))

# prepare data

target_files = os.listdir(f"{dataroot}/{target_dir}")
corpus_files = os.listdir(f"{dataroot}/{corpus_dir}")

# load in all waveforms
target_waveforms = [load_mono((f"{dataroot}/{target_dir}/{file}").to(device), target_sr) for file in target_files if file[-4:] == ".wav"]
corpus_waveforms = [load_mono((f"{dataroot}/{corpus_dir}/{file}").to(device), target_sr) for file in corpus_files if file[-4:] == ".wav"]

X_dataset = torch.zeros((0, block_length_in_samples), device=device)
for waveform in target_waveforms:
    # trim waveform to whole number of block lengths
    waveform = waveform[:,:((waveform.shape[1]//block_length_in_samples)*block_length_in_samples)]

    # reshape waveform into blocks
    blocks = torch.reshape(waveform, (-1, block_length_in_samples))
    X_dataset = torch.cat((X_dataset, blocks), dim=0)
X_dataset = X_dataset.unsqueeze(1)

Y_dataset = torch.zeros((0, block_length_in_samples), device=device)
for waveform in corpus_waveforms:
    # trim waveform to whole number of block lengths
    waveform = waveform[:,:((waveform.shape[1]//block_length_in_samples)*block_length_in_samples)]

    # reshape waveform into blocks
    blocks = torch.reshape(waveform, (-1, block_length_in_samples))
    Y_dataset = torch.cat((Y_dataset, blocks), dim=0)
Y_dataset = Y_dataset.unsqueeze(1)

# derive block lengths for discriminator
with torch.inference_mode():
    dummy_frame = dac_model.encode(X_dataset[0].unsqueeze(0))[0]
    block_length_in_frames = dummy_frame.shape[2]
    output_block_length_in_samples = dac_model.decode(dummy_frame).shape[2]

# display(Audio(target_in_concat[105,:].detach().cpu().numpy(), rate=target_sr))
    

In [9]:
gen_model = BlockGenerator().to(device)
# loss = torch.cdist()
print(gen_model)
discr_model = BlockDiscriminator(output_block_length_in_samples, block_length_in_frames).to(device)
print(discr_model)

BlockGenerator(
  (main): Sequential(
    (0): Conv1d(1024, 256, kernel_size=(5,), stride=(1,), padding=same, padding_mode=reflect)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Conv1d(256, 64, kernel_size=(5,), stride=(1,), padding=same, padding_mode=reflect)
    (3): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (4): LeakyReLU(negative_slope=0.01)
    (5): Conv1d(64, 256, kernel_size=(5,), stride=(1,), padding=same, padding_mode=reflect)
    (6): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): LeakyReLU(negative_slope=0.01)
    (8): Conv1d(256, 1024, kernel_size=(5,), stride=(1,), padding=same, padding_mode=reflect)
  )
)
BlockDiscriminator(
  (embedding_path): Sequential(
    (0): Conv1d(1024, 256, kernel_size=(4,), stride=(2,), padding=(1,))
    (1): LeakyReLU(negative_slope=0.01)
    (2): Conv1d(256, 64, kernel_size=(4,), stride=(2,), padding=(1,))
    (3): BatchNorm1d(64, eps=1e-05, momentum=0.1, a

In [10]:
class PairedWaveformDataset(torch.utils.data.Dataset):
    def __init__(self, target_data, corpus_data):
        if target_data.shape != corpus_data.shape:
            raise Exception(f"Target dataset and corpus dataset must have the same size (target dataset has shape {target_data.shape}, while corpus dataset has shape {corpus_data.shape})")
        self.target_data = target_data
        self.corpus_data = corpus_data
    
    def __len__(self):
        return self.target_data.shape[0]
    
    def __getitem__(self, idx):
        x = self.target_data[idx]
        y = self.corpus_data[idx]
        return x, y

In [11]:
paired_dataset = PairedWaveformDataset(X_dataset, Y_dataset)

dataloader = torch.utils.data.DataLoader(paired_dataset, batch_size=batch_size, shuffle=True)

In [12]:
torch.autograd.set_detect_anomaly(True)

# training
embedding_loss_fn = nn.MSELoss()
adversarial_loss_fn = nn.BCELoss()
lambda_embedding = 100
epochs = 10

gen_optimizer = optim.Adam(gen_model.parameters(), lr=0.00002, betas=(0.5, 0.999))
discr_optimizer = optim.Adam(discr_model.parameters(), lr=0.00002, betas=(0.5, 0.999))

real_label = 1
fake_label = 0

for i in range(epochs):
    # print("*", end="")
    for batch_nr, (X, Y) in enumerate(dataloader):
        print(f"Epoch: {i+1}/{epochs}, Batch: {batch_nr+1}/{len(dataloader)}")

        with torch.no_grad():
            Z_x = dac_model.encode(X)[0]
            Z_y = dac_model.encode(Y)[0]

        # train discriminator
        Z_y_pred = gen_model(Z_x).detach()
        
        with torch.no_grad():
            Y_pred = dac_model.decode(Z_y_pred)

        Y = Y[:,:,:Y_pred.shape[2]] # trim tail of y that is lost when decoding
        
        d_real = discr_model(Z_x, Y)
        d_fake = discr_model(Z_x, Y_pred)

        real_labels = torch.full(d_real.shape, real_label, device=device, dtype=torch.float32)
        real_adversarial_loss = adversarial_loss_fn(d_real, real_labels)

        fake_labels = torch.full(d_fake.shape, fake_label, device=device, dtype=torch.float32)
        fake_adversarial_loss = adversarial_loss_fn(d_fake, fake_labels)

        discr_loss = real_adversarial_loss + fake_adversarial_loss
        discr_optimizer.zero_grad()
        discr_loss.backward(retain_graph=True)
        discr_optimizer.step()

        # train generator
        
        Z_y_pred = gen_model(Z_x)
        embedding_loss = embedding_loss_fn(Z_y_pred, Z_y)

        d_fake = discr_model(Z_x, Y_pred)
        fake_adversarial_loss = adversarial_loss_fn(d_fake, fake_labels)

        gen_loss = 1/fake_adversarial_loss + lambda_embedding * embedding_loss
        gen_optimizer.zero_grad()
        gen_loss.backward()
        gen_optimizer.step()

Epoch: 1/10, Batch: 1/176
Epoch: 1/10, Batch: 2/176
Epoch: 1/10, Batch: 3/176
Epoch: 1/10, Batch: 4/176
Epoch: 1/10, Batch: 5/176
Epoch: 1/10, Batch: 6/176
Epoch: 1/10, Batch: 7/176
Epoch: 1/10, Batch: 8/176
Epoch: 1/10, Batch: 9/176
Epoch: 1/10, Batch: 10/176
Epoch: 1/10, Batch: 11/176
Epoch: 1/10, Batch: 12/176
Epoch: 1/10, Batch: 13/176
Epoch: 1/10, Batch: 14/176
Epoch: 1/10, Batch: 15/176
Epoch: 1/10, Batch: 16/176
Epoch: 1/10, Batch: 17/176
Epoch: 1/10, Batch: 18/176
Epoch: 1/10, Batch: 19/176
Epoch: 1/10, Batch: 20/176
Epoch: 1/10, Batch: 21/176
Epoch: 1/10, Batch: 22/176
Epoch: 1/10, Batch: 23/176
Epoch: 1/10, Batch: 24/176
Epoch: 1/10, Batch: 25/176
Epoch: 1/10, Batch: 26/176
Epoch: 1/10, Batch: 27/176
Epoch: 1/10, Batch: 28/176
Epoch: 1/10, Batch: 29/176
Epoch: 1/10, Batch: 30/176
Epoch: 1/10, Batch: 31/176
Epoch: 1/10, Batch: 32/176
Epoch: 1/10, Batch: 33/176
Epoch: 1/10, Batch: 34/176
Epoch: 1/10, Batch: 35/176
Epoch: 1/10, Batch: 36/176
Epoch: 1/10, Batch: 37/176
Epoch: 1/1

In [15]:
torch.save({"gen_model": gen_model.state_dict(), "discr_model": discr_model.state_dict(), "block_length_in_samples": output_block_length_in_samples, "block_length_in_frames": block_length_in_frames}, "test_checkpoint.pth")

In [ ]:
checkpoint = torch.load("test_checkpoint.pth", weights_only=True)

gen_model = BlockGenerator()
gen_model.load_state_dict(checkpoint["gen_model"])
discr_model = BlockDiscriminator(checkpoint["block_length_in_samples"], checkpoint["block_length_in_frames"])
gen_model.load_state_dict(checkpoint["discr_model"])

In [ ]:
gen_model.eval()
discr_model.eval()

test_file = "training_data/beatbox/8.wav"

test_waveform = load_mono(test_file, target_sr).to(device)
# trim waveform to whole number of block lengths
test_waveform = test_waveform[:,:((test_waveform.shape[1]//block_length_in_samples)*block_length_in_samples)]

# reshape waveform into blocks
test_blocks = torch.reshape(test_waveform, (-1, block_length_in_samples))

with torch.inference_mode():
    test_embeddings = dac_model.encode(test_blocks.unsqueeze(1))[0]

    transformed_test_embeddings = gen_model(test_embeddings)

    test_reconstruction = dac_model.decode(transformed_test_embeddings)

test_reconstructed_audio = test_reconstruction.flatten()


In [ ]:
display(Audio(test_waveform[0].detach().cpu().numpy(), rate=target_sr))
display(Audio(test_reconstructed_audio.detach().cpu().numpy(), rate=target_sr))

: 